In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "07-application-agent-framework/retrieval-rag/rag-from-scratch/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 00 · Setup & the corpus

RAG has exactly two parts that move. It must **retrieve** the relevant text, and
then **generate** an answer that depends on that text. This repo teaches the
retrieval half. You build that half from scratch, because almost all RAG
failures occur in it.

**What you need**

```
pip install -r requirements.txt        # T0: numpy + pytest, no torch
pip install -r requirements-full.txt   # optional: sentence-transformers (and torch) for the real models
```

If you do not have `sentence-transformers`, `ragkit.embed` uses a labelled hashing
embedder as a fallback. This embedder is lexical, not semantic. Every notebook
still runs. If you have `sentence-transformers`, the first embedding call downloads
a ~90 MB model (`all-MiniLM-L6-v2`). Then the model runs offline on the CPU.

**Generation** is optional. To use a real model, set `ANTHROPIC_API_KEY` or
`OPENAI_API_KEY`. If you do not set a key, a deterministic *extractive* fallback
answers from the retrieved text. Thus every notebook still runs.


In [ ]:
# --- setup: make `import ragkit` work from notebooks/ or solutions/ ---
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
from ragkit.corpus import load_documents, load_corpus, load_qrels, tokenize
from ragkit.embed import get_embedder
from ragkit import llm

In [ ]:
# Which generation backend is active?
print("generation backend:", llm.available())

# Is the embedding model installed?
try:
    emb = get_embedder()
    print("embedder ready:", emb.model_name, "| dim =", emb.dim)
except ImportError as e:
    print("EMBEDDER NOT INSTALLED — run: pip install sentence-transformers\n")
    print(e)

## The corpus

The corpus has nine short Markdown docs for a fictional SaaS company, "Meridian".
Each doc has `##` sections (that structure is important in notebook 02). A small
labelled question set (`qrels`) lets us *measure* retrieval in notebook 05.


In [ ]:
docs = load_documents()
print(len(docs), "documents:\n", ", ".join(docs))

print("\n--- sample document: expense-policy ---\n")
print(docs["expense-policy"])

In [ ]:
qrels = load_qrels()
from collections import Counter
print("questions by kind:", dict(Counter(q["kind"] for q in qrels)), "\n")
for kind in ("lexical", "semantic", "multihop"):
    ex_q = next(q for q in qrels if q["kind"] == kind)
    print(f"[{kind:8}] {ex_q['question']}\n           gold -> {ex_q['gold_docs']}")

## The one idea behind dense retrieval

An **embedding model** maps text to a vector, so that *texts with similar meanings
get vectors that are near each other*. Thus "retrieval" is only two steps: embed
the query, then find the nearest document vectors. The next cell shows this
directly.


In [ ]:
# (needs the embedder installed)
pairs = [
    ("How many holidays do I get?", "annual leave and vacation days"),   # related
    ("How many holidays do I get?", "the API returns HTTP 429 when throttled"),  # unrelated
]
for a, b in pairs:
    va, vb = get_embedder().encode(a), get_embedder().encode(b)
    print(f"cos = {float(va @ vb):+.3f}   {a!r}  vs  {b!r}")
# Rows are L2-normalised, so the dot product IS cosine similarity.

**Learning path**

| nb | idea |
|----|------|
| 01 | the minimal RAG loop: embed, cosine search, prompt, generate |
| 02 | chunking: why *how* you divide the text decides what you can retrieve |
| 03 | hybrid search: BM25 (exact) and dense (meaning), fused with RRF |
| 04 | reranking: low-cost recall, then a cross-encoder for precision |
| 05 | evaluation: hit@k, recall@k and MRR. Measure everything from 01 to 04. |
| 06 | iterative RAG (advanced): multi-hop questions need >1 retrieval |

Do the exercises in each notebook. Fill in the `# YOUR CODE HERE` blanks. The
`assert`s examine your work. The solutions are in `solutions/`.
